# Model Evaluation

Prophet, XGBoost, and LightGBM against a naive baseline on ERCOT day-ahead SPP. Same data, split, features, and champion-selection metric as `src/train.py` — this notebook calls those functions directly instead of reimplementing them.

## Setup

In [ ]:
from src.config import load_config
from src.train import (
    load_timeseries_data,
    split_train_test,
    train_prophet,
    train_xgboost,
    train_lightgbm,
    calculate_baseline_mae,
    select_champion,
    get_model_mae,
)

config = load_config()

## Load data

`load_timeseries_data` — SPP target joined with the leakage-guarded load-forecast feature.

In [ ]:
df = load_timeseries_data(config)
print(f"{df.height} rows, {df['ts'].min()} to {df['ts'].max()}")
df.head()

## Explore

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plt.plot(df["ts"].to_list(), df["value"].to_list(), linewidth=1)
plt.title(f"{config.settlement_point} {config.forecast_measurement} ({config.forecast_unit})")
plt.xlabel("Time")
plt.ylabel(config.forecast_unit)
plt.tight_layout()
plt.show()

## Train/test split

Time-ordered, last 30 days held out — `split_train_test`.

In [ ]:
train_df, test_df = split_train_test(df)
print(f"train: {train_df.height} rows | test: {test_df.height} rows")

## Baseline + three challengers

In [ ]:
baseline_mae = calculate_baseline_mae(test_df)
prophet_mae, prophet_model = train_prophet(train_df, test_df, config)
xgboost_mae, xgboost_model = train_xgboost(train_df, test_df)
lightgbm_mae, lightgbm_model = train_lightgbm(train_df, test_df)

print(f"baseline (mean):  {baseline_mae:.2f}")
print(f"prophet:          {prophet_mae:.2f}")
print(f"xgboost:          {xgboost_mae:.2f}")
print(f"lightgbm:         {lightgbm_mae:.2f}")

## Champion

Selected by MAE alone — `select_champion`. First run, no current champion to beat.

In [ ]:
champion = select_champion(prophet_mae, xgboost_mae, lightgbm_mae, current_champion=None)
champion_mae = get_model_mae(champion, prophet_mae, xgboost_mae, lightgbm_mae)
beats_baseline = champion_mae < baseline_mae

print(f"champion: {champion.value} (MAE={champion_mae:.2f})")
print(f"beats baseline: {beats_baseline}")

## Comparison

In [ ]:
import polars as pl

comparison = pl.DataFrame({
    "model": ["baseline", "prophet", "xgboost", "lightgbm"],
    "mae": [baseline_mae, prophet_mae, xgboost_mae, lightgbm_mae],
})
print(comparison)

plt.figure(figsize=(8, 5))
plt.bar(comparison["model"].to_list(), comparison["mae"].to_list())
plt.ylabel("MAE")
plt.title("Holdout MAE by model")
plt.tight_layout()
plt.show()